In [1]:
import numpy as np
import pandas as pd
import torch
import time
import json

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


/home/protein/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
train_df = pd.read_csv("PI_nonPI_train_80.csv")
test_df = pd.read_csv("PI_nonPI_test_20.csv")

train_sequences = train_df["sequence"].tolist()
train_labels = train_df["label"].tolist()
test_sequences = test_df["sequence"].tolist()
test_labels = test_df["label"].tolist()


In [3]:
tokenizer = BertTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

max_len = 250

train_encodings = tokenizer(
    train_sequences,
    truncation=True,
    padding=True,
    max_length=max_len,
    return_tensors="pt"
)

test_encodings = tokenizer(
    test_sequences,
    truncation=True,
    padding=True,
    max_length=max_len,
    return_tensors="pt"
)



In [4]:
train_inputs = train_encodings.input_ids.to(device)
train_masks = train_encodings.attention_mask.to(device)
test_inputs = test_encodings.input_ids.to(device)
test_masks = test_encodings.attention_mask.to(device)

train_labels = torch.tensor(train_labels, dtype=torch.long).to(device)
test_labels = torch.tensor(test_labels, dtype=torch.long).to(device)


In [5]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_labels.cpu().numpy()
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)


In [6]:
model = BertForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=2
).to(device)

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

batch_size = 16
epochs = 3


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
model.train()
start_time = time.time()

for epoch in range(epochs):
    for i in range(0, len(train_inputs), batch_size):
        batch_inputs = train_inputs[i:i + batch_size]
        batch_masks = train_masks[i:i + batch_size]
        batch_labels = train_labels[i:i + batch_size]

        optimizer.zero_grad()
        outputs = model(input_ids=batch_inputs, attention_mask=batch_masks)
        loss = loss_fct(outputs.logits, batch_labels)
        loss.backward()
        optimizer.step()

end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} seconds")


Training time: 458.66 seconds


In [8]:
save_dir = "biobert_model"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

label_info = {
    "id2label": {0: "negative", 1: "positive"},
    "label2id": {"negative": 0, "positive": 1}
}
with open(f"{save_dir}/labels.json", "w") as f:
    json.dump(label_info, f, indent=2)


In [9]:
model.eval()
with torch.no_grad():
    outputs = model(test_inputs, attention_mask=test_masks)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)[:, 1]
    predictions = torch.argmax(logits, dim=1)

y_true = test_labels.cpu().numpy()
y_pred = predictions.cpu().numpy()
y_prob = probs.cpu().numpy()


In [10]:
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Performance Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"MCC:       {mcc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")


Performance Metrics:
Accuracy:  0.7753
Precision: 0.8768
Recall:    0.3378
F1 Score:  0.4877
ROC-AUC:   0.6762
MCC:       0.4488
Sensitivity: 0.3378
Specificity: 0.9780
